## 7. 现代卷积神经网络

- 卷积神经网络可以说是面向计算机视觉的一个架构，而提到计算机视觉，绕不开的一个话题就是ImageNet竞赛

- 本章主要围绕一些“现代”的卷积神经网络架构，之所以给“现代”这个词打上引号，很大的原因在于这些架构都是在d2l成书当时比较流形和熟悉的架构，在这些年当中，这个领域又出现了很多其他的工作

- 本章主要介绍的模型包括：
    - AlexNet，这是第一个在ImageNet中击败传统计算机视觉模型的大型神经网络
    - VGG，利用许多重复的神经网络块
    - NiN，重复使用卷积层和1*1卷积层构建深层网络
    - GoogleNet，使用并行连结的网络，通过不同大小的卷积核并行抽取信息
    - ResNet，通过残差块构建跨层的数据通道
    - DenseNet，计算成本高但效果更好的网络设计
    - 批量规范化(batch normalization)技术的使用

### 7.1 深度卷积神经网络(AlexNet)

- LeNet虽然效果优异，但成功建立在小数据集上，在更大、更真实的数据集中并未得到证实

- 相较于传统的机器学习，推崇卷积神经网络类似架构的计算机视觉研究人员思考问题的角度有所不同：
    - 传统机器学习的pipeline类似：
        - 收集一个具有针对性的数据集
        - 手工对特征数据集进行预处理
        - 通过标准的特征提取算法处理数据，换言之传统机器学习方法输入模型的一定不是raw data
        - 根据理论证明模型的性质，在特定的模型中训练分类器
    - 但CV的研究人员更笃定大而干净的数据集、优秀的数据提取远比优雅的理论证明、学习算法的改进带来的进步大得多

#### 7.1.1 学习表征

- 说明这两条路线一个很好的解决思路就是查看图像特征的提取方法
    - 在2012年前,图像特征都是机械地计算出来的。事实上,设计一套新的特征函数、改进结果,并撰写论文是盛极一时的潮流。SIFT(Lowe, 2004)、SURF(Bay et al., 2006)、HOG(定向梯度直方图)(Dalal and Triggs, 2005)、bags of visual words和类似的特征提取方法占据了主导地位
    - Yann LeCun、Geoff Hinton、Yoshua Bengio、Andrew Ng、Shun ichi Amari和Juergen Schmidhuber等则看法不同：**特征本身应该被学习**
        - 此外，特征还应该由多个共同学习的神经网络层组成，每个层都有学习的参数
        - 模型的底层应该尽可能地去检测边缘、颜色和纹理等信息(更靠近输入的部分)
    - AlexNet在2012年的ImageNet挑战赛中取得了非常好的成绩，在网络的底层就学习到了一些类似传统滤波器的特征提取器，如下图
        - ![7.1](./images/chapter7/1.png)
    - AlexNet的更高层学习到的特征更为高级，类似围绕一个图像中的具体元素等等

- 类似的路线一直有人钻研，但最终在2012年的时间节点由AlexNet突破，d2l中将这个突破归因于两个关键因素
    - 数据：
        - 包含许多特征的深度模型需要大量的有标签数据，但早期的大部分研究只基于小型的公开数据集
        - 2009年,ImageNet数据集发布,并发起ImageNet挑战赛:要求研究人员从100万个样本中训练模型,以区分1000个不同类别的对象。
            - ImageNet数据集由斯坦福教授李飞飞小组的研究人员开发,利用谷歌图像搜索(Google Image Search)对每一类图像进行预筛选,并利用亚马逊众包(Amazon Mechanical Turk)来标注每张图片的相关类别
    - 硬件：
        - 深度学习对计算资源要求很高，因此在机器学习的早期研究人员更倾向于优化凸目标的简单算法
        - GPU的出现改变了这一格局
            - CPU中每个核心都具有高时钟频率的运行能力和L3Cache，但正因为CPU可以执行各种指令，通用核心的特点也使得在单个任务上的性能相对较差
            - GPU中有上千个处理单元(目前Hopper架构和Blackwell架构的tensor核心已是两万左右)，这些处理单元虽然单个核心的时钟频率较低，但庞大的数量和成组(warps)的计算方式使GPU的浮点性能相较于CPU大幅提升
    - 回到2012年的重大突破,当Alex Krizhevsky和Ilya Sutskever实现了可以在GPU硬件上运行的深度卷积神经网络时,一个重大突破出现了。他们意识到卷积神经网络中的计算瓶颈:卷积和矩阵乘法,都是可以在硬件上并行化的操作。于是,他们使用两个显存为3GB的NVIDIA GTX580 GPU实现了快速卷积运算。

#### 7.1.2 AlexNet

- AlexNet和LeNet的架构非常类似，d2l中呈现的是一个简化版本，原始的AlexNet因为需要在两张卡上并行运算，因此有一些其他的组件

- 简化的AlexNet和LeNet的结构对比如图：
    - ![7.2](./images/chapter7/2.png)

- AlexNet比相对较小的LeNet5要深得多。AlexNet由八层组成:五个卷积层、两个全连接隐藏层和一个全连接输出层；AlexNet使用ReLU而不是sigmoid作为激活函数

- 这张图和这里介绍的AlexNet，最大的不同就是真实的实现中的第一个和第二个卷积层都是在两张GPU上并行均分的，在通道维度上做了切分以便能在GPU上顺利运行

- 模型设计
    - 